# Stage 8: exporting XGBoost + HMM to the watch (C code)

The recommended model (Stages 4–6) is XGBoost + HMM. This stage turns it into C code that runs on the watch's ESP32-C3:

1. **Make it fit.** The Stage 3 model is about 237,000 tree nodes, about 2 MB as C tables. Too big. Pick a smaller one, using training people only
2. **Test the smaller model**, and a **live HMM** that only uses the past (the watch can't see the future)
3. **Export**: generate the C tables, next to the hand-written `export/sg_xgb.h` (features, trees, live HMM)
4. **Prove the C gives the same answers as Python**: compile it on this laptop and compare, window by window
5. How to add it to the firmware

Training runs on the **RTX 3060** only. **Kernel:** `.venv-tf` in this folder.

## How this works (for beginners)

**What "exporting" means.** In Python, the model is an object in memory. The watch has no Python: it runs C code. So the model's trees are written out as **tables of numbers** in a C file, plus a small C function that walks through them. The same goes for the 16 features: they're recomputed in C, from the watch's own 30 s of samples.

**Why size matters.** The ESP32-C3 SuperMini has **4 MB of flash** for *everything*: Wi-Fi, secure connections, audio, the fall model, and this model. A model is mostly its tree nodes. Each node is a question ("is feature 3 below 0.12?") and takes about 9 bytes, so:

| Model | Trees | Nodes | Size |
|---|---|---|---|
| Stage 3 XGBoost (depth 8) | 800 | ~237,000 | ~2 MB |
| depth 4, for example | 800 | ~24,000 | ~0.2 MB |

**Smaller trees ask fewer questions**, so they can be a bit less accurate. Section 1 measures how much.

**No floating-point hardware.** The ESP32-C3 does decimal maths in software, slowly. The C code uses `float` (not `double`) in the heavy parts. Section 4 checks this doesn't change the answers.

**Live HMM.** The notebooks' HMM (Viterbi) picks the best labels for a whole day, using the future too. On the watch, each new 30 s window has to be labelled *now*. The **forward filter** keeps a running belief ("given everything so far, how likely is each activity?") and updates it with each new window. Section 2 measures what that costs.

In [1]:
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_sample_weight

from common import (CLASSES, DATA_DIR, FEATURES, HIGH_BAND, LOW_BAND, RESULTS_DIR, SEED, make_features, paired_bootstrap,
                    scores, smooth, split_people, to_binary)

assert xgb.build_info()['USE_CUDA'], 'this XGBoost cannot use the GPU'
EXPORT = Path('export')
f = np.load(DATA_DIR / 'features_all.npz')
F, y, pid = f['F'], f['y'], f['pid']
train_ids, test_ids = split_people()
tr, te = np.isin(pid, train_ids), np.isin(pid, test_ids)
F_tr, y_tr, pid_tr, F_te, y_te, pid_te = F[tr], y[tr], pid[tr], F[te], y[te], pid[te]
stage3 = np.load(RESULTS_DIR / 'stage3_xgb_predictions.npz')
assert (stage3['y'] == y_te).all() and (stage3['pid'] == pid_te).all()   # same test windows, same order
hmm = np.load(RESULTS_DIR / 'hmm.npz')
print(f'train {len(y_tr):,} windows ({len(train_ids)} people), test {len(y_te):,} ({len(test_ids)} people)')

train 243,012 windows (121 people), test 58,152 (30 people)


## 1. Make it fit: choose a smaller model, using training people only

**Budget:** the firmware already holds Wi-Fi, secure connections, audio and the fall model. The default Arduino setting gives the program about 1.2 MB, so the activity model should stay **under 200 kB**.

**The rule, decided before looking:** among the sizes below, take the **smallest** whose 5-fold validation macro-F1 (grouped by person, training people only, as in Stage 3) is within **0.005** of the Stage 3 model's 0.740. If none qualifies, the best one under 200 kB.

**Bytes per node:** threshold or leaf value (4) + which feature (1) + left and right child (2 + 2, or 4 + 4 above 65,535 nodes).

In [2]:
def make_xgb(depth, n_trees):
    return xgb.XGBClassifier(n_estimators=n_trees, max_depth=depth, learning_rate=0.1, objective='multi:softprob',
                             tree_method='hist', device='cuda', n_jobs=4, random_state=SEED)

def fit(model, F_, y_):
    return model.fit(F_, y_, sample_weight=compute_sample_weight('balanced', y_))

def model_bytes(booster):
    trees = booster.get_booster().trees_to_dataframe()
    n_nodes, n_trees = len(trees), trees['Tree'].nunique()
    index = 2 if n_nodes < 65536 else 4
    return n_nodes, n_nodes * (4 + 1 + 2 * index) + n_trees * index

stage3_f1 = pd.read_csv(RESULTS_DIR / 'stage3_xgb_tuning.csv').set_index(['max_depth', 'n_estimators']).loc[(8, 200), 'mean macro-F1']
rows = []
t0 = time.time()
for depth in [3, 4, 5, 6]:
    for n_trees in [100, 200, 300]:
        f1s, sizes = [], []
        for fit_idx, val_idx in GroupKFold(5).split(F_tr, y_tr, pid_tr):
            m = fit(make_xgb(depth, n_trees), F_tr[fit_idx], y_tr[fit_idx])
            f1s.append(scores(y_tr[val_idx], m.predict(F_tr[val_idx]), 4)['macro-F1'])
            sizes.append(model_bytes(m))
        rows.append({'max_depth': depth, 'rounds': n_trees, 'mean macro-F1': np.mean(f1s),
                     'nodes': int(np.mean([s[0] for s in sizes])), 'kB': np.mean([s[1] for s in sizes]) / 1000})
    print(f'depth {depth} done, {time.time() - t0:.0f} s')
grid = pd.DataFrame(rows)
grid.to_csv(RESULTS_DIR / 'stage8_size_grid.csv', index=False)
print(f'\nStage 3 model (depth 8, 200 rounds): macro-F1 {stage3_f1:.3f}')
print(grid.round(3).to_string(index=False))

ok = grid[(grid['mean macro-F1'] >= stage3_f1 - 0.005) & (grid.kB < 200)]
choice = ok.nsmallest(1, 'kB').iloc[0] if len(ok) else grid[grid.kB < 200].nlargest(1, 'mean macro-F1').iloc[0]
DEPTH, ROUNDS = int(choice.max_depth), int(choice.rounds)
print(f'\nchosen: depth {DEPTH}, {ROUNDS} rounds (macro-F1 {choice["mean macro-F1"]:.3f}, about {choice.kB:.0f} kB)')

C:\Users\syafi\SilverGuard Major Project\notebooks\DEEPDEEPDEEP_learning\.venv-tf\Lib\site-packages\xgboost\core.py:751: UserWarning: [21:40:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


depth 3 done, 26 s


depth 4 done, 56 s


depth 5 done, 96 s


depth 6 done, 151 s

Stage 3 model (depth 8, 200 rounds): macro-F1 0.740
 max_depth  rounds  mean macro-F1  nodes       kB
         3     100          0.719   5943   54.292
         3     200          0.726  11779  107.616
         3     300          0.730  17640  161.160
         4     100          0.726  11936  108.224
         4     200          0.733  23545  213.507
         4     300          0.736  35107  318.365
         5     100          0.732  23295  210.457
         5     200          0.737  45030  406.874
         5     300          0.738  66739  872.415
         6     100          0.736  43610  393.294
         6     200          0.738  81402 1061.426
         6     300          0.737 119238 1554.904

chosen: depth 3, 300 rounds (macro-F1 0.730, about 161 kB)


## 2. Train it, and test it the same way as before

Trained on the 121 training people (GPU), scored on the 30 test people, next to the Stage 3 model. Both with the notebooks' whole-day HMM (Viterbi) and with the watch's **live** HMM.

In [3]:
model = fit(make_xgb(DEPTH, ROUNDS), F_tr, y_tr)
model.save_model(RESULTS_DIR / 'xgb_watch.json')
probs_small = model.predict_proba(F_te)

def live_hmm(probs, pids, start, transition):
    """The forward filter, as in sg_xgb.h: each window labelled using only the windows before it."""
    out = np.zeros(len(probs), dtype=int)
    for p in np.unique(pids):
        idx = np.flatnonzero(pids == p)
        belief = None
        for i in idx:
            e = np.clip(probs[i], 1e-6, 1)
            belief = start * e if belief is None else (belief @ transition) * e
            belief /= belief.sum()
            out[i] = belief.argmax()
    return out

preds = {}
for name, pr in [('Stage 3 XGB (big)', stage3['probs']), ('watch XGB (small)', probs_small)]:
    preds[f'{name}'] = pr.argmax(axis=1)
    preds[f'{name} + HMM (whole day)'] = smooth(pr, pid_te, hmm['start'], hmm['transition'])
    preds[f'{name} + live HMM'] = live_hmm(pr, pid_te, hmm['start'], hmm['transition'])
results = pd.DataFrame([{'model': n, '4-class kappa': scores(y_te, p, 4)['kappa'], '4-class macro-F1': scores(y_te, p, 4)['macro-F1'],
                         'binary kappa': scores(to_binary(y_te), to_binary(p), 2)['kappa']} for n, p in preds.items()])
results.to_csv(RESULTS_DIR / 'stage8_test_scores.csv', index=False)
print(results.round(3).to_string(index=False))

per = pd.DataFrame({n: [scores(y_te[pid_te == p], v[pid_te == p], 4)['kappa'] for p in test_ids] for n, v in preds.items()})
for a, b in [('watch XGB (small) + HMM (whole day)', 'Stage 3 XGB (big) + HMM (whole day)'),
             ('watch XGB (small) + live HMM', 'watch XGB (small) + HMM (whole day)')]:
    mean, lo, hi = paired_bootstrap(per[a], per[b])
    print(f'\n{a}  vs  {b}: mean 4-class kappa difference {mean:+.3f} (95% range {lo:+.3f} to {hi:+.3f})')

                              model  4-class kappa  4-class macro-F1  binary kappa
                  Stage 3 XGB (big)          0.696             0.741         0.697
Stage 3 XGB (big) + HMM (whole day)          0.818             0.823         0.790
       Stage 3 XGB (big) + live HMM          0.793             0.800         0.760
                  watch XGB (small)          0.686             0.727         0.684
watch XGB (small) + HMM (whole day)          0.810             0.816         0.767
       watch XGB (small) + live HMM          0.782             0.789         0.739



watch XGB (small) + HMM (whole day)  vs  Stage 3 XGB (big) + HMM (whole day): mean 4-class kappa difference -0.007 (95% range -0.022 to +0.007)

watch XGB (small) + live HMM  vs  watch XGB (small) + HMM (whole day): mean 4-class kappa difference -0.029 (95% range -0.038 to -0.020)


## 3. Export: write the C tables

The trees become flat tables: for node *n*, `SG_NODE_FEATURE[n]` (255 = leaf), `SG_NODE_VALUE[n]` (threshold, or the leaf's score), `SG_NODE_LEFT[n]` and `SG_NODE_RIGHT[n]`. XGBoost goes **left when feature < threshold**.

First, a check in Python that walking these exact tables gives exactly XGBoost's answers.

In [4]:
def tables_from(model, rounds):
    """The model's trees as flat tables, exactly as the C code walks them, checked against XGBoost."""
    trees = model.get_booster().trees_to_dataframe()
    assert (trees.groupby('Tree').head(1)['Node'] == 0).all()
    n_trees = trees['Tree'].nunique()
    assert n_trees == rounds * len(CLASSES)
    node_ids = {nid: i for i, nid in enumerate(trees['ID'])}           # '3-7' (tree-node) -> row in the flat table
    is_leaf = trees['Feature'] == 'Leaf'
    t = {'n_trees': n_trees,
         'feature': np.where(is_leaf, 255, [int(str(v).lstrip('f')) if not leaf else 0
                                            for v, leaf in zip(trees['Feature'], is_leaf)]).astype(np.uint8),
         'value': np.where(is_leaf, trees['Gain'], trees['Split']).astype(np.float32),
         'left': np.array([node_ids[v] if isinstance(v, str) else 0 for v in trees['Yes']]),
         'right': np.array([node_ids[v] if isinstance(v, str) else 0 for v in trees['No']]),
         'roots': np.array([node_ids[f'{k}-0'] for k in range(n_trees)])}
    F32 = F_te.astype(np.float32)
    tree_sum = walk(F32, t)
    t['base_margin'] = (model.get_booster().predict(xgb.DMatrix(F32), output_margin=True) - tree_sum).mean(axis=0).astype(np.float32)
    margin = tree_sum + t['base_margin']
    p = np.exp(margin - margin.max(axis=1, keepdims=True)); p /= p.sum(axis=1, keepdims=True)
    t['gap'] = np.abs(p - model.predict_proba(F_te)).max()
    assert t['gap'] < 1e-5, t['gap']
    return t

def walk(F32, t):
    """The exact steps of sg_predict_proba, in numpy float32 (before the starting margin is added)."""
    margin = np.zeros((len(F32), len(CLASSES)), dtype=np.float32)
    for k, r in enumerate(t['roots']):
        n = np.full(len(F32), r)
        while True:
            inner = t['feature'][n] != 255
            if not inner.any():
                break
            go_left = F32[np.arange(len(F32)), np.where(inner, t['feature'][n], 0)] < t['value'][n]
            n = np.where(inner, np.where(go_left, t['left'][n], t['right'][n]), n)
        margin[:, k % len(CLASSES)] += t['value'][n]
    return margin

tab = tables_from(model, ROUNDS)
value = tab['value']
print(f"{len(value):,} nodes in {tab['n_trees']} trees. Walking the tables gives XGBoost's probabilities to within {tab['gap']:.1e}")

17,658 nodes in 1200 trees. Walking the tables gives XGBoost's probabilities to within 2.4e-07


In [5]:
def c_floats(a):
    return ', '.join(f'{repr(float(np.float32(v)))}f' for v in np.ravel(a))

def c_ints(a):
    return ', '.join(str(int(v)) for v in np.ravel(a))

low_k, high_k = np.flatnonzero(LOW_BAND), np.flatnonzero(HIGH_BAND)
assert high_k.min() == low_k.max() + 1                                # the two bands sit side by side

def write_header(t, hmm_start, hmm_transition, trained_on):
    """Write export/sg_xgb_model.h. -> size in the watch's flash (kB)."""
    index_type = 'uint16_t' if len(t['value']) < 65536 else 'uint32_t'
    header = f"""// GENERATED by stage8_export_xgboost.ipynb. Do not edit: re-run the notebook.
// XGBoost depth {DEPTH}, {ROUNDS} rounds x {len(CLASSES)} classes = {t['n_trees']} trees, {len(t['value'])} nodes.
// {trained_on}
#ifndef SG_XGB_MODEL_H
#define SG_XGB_MODEL_H
#include <stdint.h>

#define SG_N_FEATURES {len(FEATURES)}
#define SG_N_CLASSES {len(CLASSES)}
#define SG_N_TREES {t['n_trees']}
#define SG_N_NODES {len(t['value'])}
#define SG_LEAF 255
#define SG_K_FIRST {low_k.min()}      // first rhythm step (0.5 Hz); step k = k/30 Hz
#define SG_K_LOW_LAST {low_k.max()}   // last step of the 0.5-3 Hz band
#define SG_K_LAST {high_k.max()}      // last step (8 Hz)
typedef {index_type} sg_index_t;

static const char *const SG_FEATURE_NAMES[SG_N_FEATURES] = {{{', '.join(f'"{n}"' for n in FEATURES)}}};
static const char *const SG_CLASS_NAMES[SG_N_CLASSES] = {{{', '.join(f'"{n}"' for n in CLASSES)}}};
static const float SG_BASE_MARGIN[SG_N_CLASSES] = {{{c_floats(t['base_margin'])}}};

static const float SG_HMM_START[SG_N_CLASSES] = {{{c_floats(hmm_start)}}};
static const float SG_HMM_TRANSITION[SG_N_CLASSES][SG_N_CLASSES] = {{
{chr(10).join('  {' + c_floats(row) + '},' for row in hmm_transition)}
}};

static const sg_index_t SG_TREE_ROOT[SG_N_TREES] = {{{c_ints(t['roots'])}}};
static const uint8_t SG_NODE_FEATURE[SG_N_NODES] = {{{c_ints(t['feature'])}}};
static const float SG_NODE_VALUE[SG_N_NODES] = {{{c_floats(t['value'])}}};
static const sg_index_t SG_NODE_LEFT[SG_N_NODES] = {{{c_ints(t['left'])}}};
static const sg_index_t SG_NODE_RIGHT[SG_N_NODES] = {{{c_ints(t['right'])}}};

#endif
"""
    (EXPORT / 'sg_xgb_model.h').write_text(header)
    idx_bytes = 2 if index_type == 'uint16_t' else 4
    return (len(t['value']) * (4 + 1 + 2 * idx_bytes) + t['n_trees'] * idx_bytes) / 1000

model_kb = write_header(tab, hmm['start'], hmm['transition'],
                        f'Trained on the {len(train_ids)} Capture-24 training people. Test (30 people): results/stage8_test_scores.csv')
print(f'export/sg_xgb_model.h written: {(EXPORT / "sg_xgb_model.h").stat().st_size / 1e6:.2f} MB of text, {model_kb:.0f} kB in the watch\'s flash')

export/sg_xgb_model.h written: 0.64 MB of text, 161 kB in the watch's flash


**Self-test vectors.** Two real windows (one still, one moving) with Python's features and probabilities, so the watch can check itself at start-up, as the current firmware does with `test_vectors.h`.

In [6]:
for tv_person in test_ids:                         # the first test person with both a still and a very active window
    with np.load(DATA_DIR / f'{tv_person}.npz') as g:
        Xp, yp = g['X'], g['y4']
    if (yp == 1).sum() > 10 and (yp == 3).sum() > 5:
        break
vectors = Xp[[int(np.flatnonzero(yp == 1)[10]), int(np.flatnonzero(yp == 3)[5])]]
v_feat = make_features(vectors)

def write_test_vectors(model):
    v_prob = model.predict_proba(v_feat)
    tv = ['// GENERATED by stage8_export_xgboost.ipynb: two Capture-24 test windows and what Python makes of them.',
          '#ifndef SG_XGB_TEST_VECTORS_H', '#define SG_XGB_TEST_VECTORS_H', '#include "sg_xgb.h"', '#define SG_N_VECTORS 2',
          'static const float SG_TV_X[SG_N_VECTORS][SG_WIN][3] = {']
    for w in vectors:
        tv.append('  {' + ', '.join('{' + c_floats(s) + '}' for s in w) + '},')
    tv += ['};', 'static const float SG_TV_FEATURES[SG_N_VECTORS][SG_N_FEATURES] = {' + ', '.join('{' + c_floats(r) + '}' for r in v_feat) + '};',
           'static const float SG_TV_PROBA[SG_N_VECTORS][SG_N_CLASSES] = {' + ', '.join('{' + c_floats(r) + '}' for r in v_prob) + '};',
           """
// Returns the largest probability error over the test vectors. Call once at start-up; it should be below 0.001.
static float sg_self_test(void) {
  static float ax[SG_WIN], ay[SG_WIN], az[SG_WIN];
  float f[SG_N_FEATURES], p[SG_N_CLASSES], worst = 0.0f;
  for (int v = 0; v < SG_N_VECTORS; v++) {
    for (int i = 0; i < SG_WIN; i++) { ax[i] = SG_TV_X[v][i][0]; ay[i] = SG_TV_X[v][i][1]; az[i] = SG_TV_X[v][i][2]; }
    sg_features(ax, ay, az, f);
    sg_predict_proba(f, p);
    for (int c = 0; c < SG_N_CLASSES; c++) {
      float e = fabsf(p[c] - SG_TV_PROBA[v][c]);
      if (e > worst) worst = e;
    }
  }
  return worst;
}
#endif
"""]
    (EXPORT / 'sg_xgb_test_vectors.h').write_text('\n'.join(tv))
    return v_prob

v_prob = write_test_vectors(model)
print('export/sg_xgb_test_vectors.h written. Expected:', dict(zip(['still window', 'moving window'], [CLASSES[i] for i in v_prob.argmax(1)])))

export/sg_xgb_test_vectors.h written. Expected: {'still window': 'sedentary', 'moving window': 'moderate-vigorous'}


## 4. Does the C give the same answers as Python?

The C files are compiled on this laptop (with Zig's C compiler) into a small test program. It reads real test windows, runs **exactly the watch's code** (features in float, the trees, the live HMM), and writes the results back. Then, window by window, against Python:

- the 16 features
- the probabilities
- the final label after the live HMM

Four whole test people (several thousand windows), so the live HMM runs over real days.

In [7]:
harness = EXPORT / 'test_harness.c'
harness.write_text(r"""
// Desktop test of the watch code: reads windows, writes features, probabilities and live-HMM labels.
#include <stdio.h>
#include <stdlib.h>
#include "sg_xgb_test_vectors.h"
int main(int argc, char **argv) {
  FILE *in = fopen(argv[1], "rb"), *out = fopen(argv[2], "wb");
  int n, person, last = -1;
  static float w[SG_WIN * 3], ax[SG_WIN], ay[SG_WIN], az[SG_WIN];
  float f[SG_N_FEATURES], p[SG_N_CLASSES];
  sg_hmm_state s;
  printf("self-test worst probability error: %g\n", sg_self_test());
  fread(&n, sizeof(int), 1, in);
  for (int k = 0; k < n; k++) {
    fread(&person, sizeof(int), 1, in);
    fread(w, sizeof(float), SG_WIN * 3, in);
    for (int i = 0; i < SG_WIN; i++) { ax[i] = w[3 * i]; ay[i] = w[3 * i + 1]; az[i] = w[3 * i + 2]; }
    if (person != last) { sg_hmm_reset(&s); last = person; }   // a new person starts a new day
    sg_features(ax, ay, az, f);
    sg_predict_proba(f, p);
    float label = (float)sg_hmm_step(&s, p);
    fwrite(f, sizeof(float), SG_N_FEATURES, out);
    fwrite(p, sizeof(float), SG_N_CLASSES, out);
    fwrite(&label, sizeof(float), 1, out);
  }
  fclose(out);
  return 0;
}
""")

check_people = test_ids[:4]
Xs, ps = [], []
for k, p in enumerate(check_people):
    with np.load(DATA_DIR / f'{p}.npz') as g:
        Xs.append(g['X']); ps.append(np.full(len(g['X']), k, dtype=np.int32))
Xs, ps = np.concatenate(Xs).astype(np.float32), np.concatenate(ps)
with open(EXPORT / 'check_in.bin', 'wb') as fh:
    np.array([len(Xs)], dtype=np.int32).tofile(fh)
    for w, k in zip(Xs, ps):
        np.array([k], dtype=np.int32).tofile(fh); w.tofile(fh)
py_feat = make_features(Xs)

def check_c(model, hmm_start, hmm_transition):
    """Compile the CURRENT export files and compare the C program with Python, window by window."""
    exe = EXPORT / 'test_harness.exe'
    build = subprocess.run([sys.executable, '-m', 'ziglang', 'cc', '-O2', '-Wall', '-Wno-unused-function', '-o', str(exe), str(harness)],
                           capture_output=True, text=True)
    assert build.returncode == 0, build.stderr
    t0 = time.time()
    run = subprocess.run([str(exe), str(EXPORT / 'check_in.bin'), str(EXPORT / 'check_out.bin')], capture_output=True, text=True)
    assert run.returncode == 0, run.stderr
    print('compiled with no warnings' if not build.stderr.strip() else build.stderr, '|', run.stdout.strip(),
          f'| {len(Xs):,} windows in {time.time() - t0:.1f} s on the laptop')
    out = np.fromfile(EXPORT / 'check_out.bin', dtype=np.float32).reshape(len(Xs), -1)
    c_feat, c_prob, c_label = out[:, :16], out[:, 16:20], out[:, 20].astype(int)
    py_prob = model.predict_proba(py_feat)
    py_label = live_hmm(py_prob, ps, hmm_start, hmm_transition)
    rel = np.abs(c_feat - py_feat) / (np.abs(py_feat) + 1e-3)
    agreement = {'largest feature difference (relative)': float(rel.max()),
                 'largest probability difference': float(np.abs(c_prob - py_prob).max()),
                 'windows differing by over 0.0001': int((np.abs(c_prob - py_prob).max(axis=1) > 1e-4).sum()),
                 'same top class (no HMM), %': 100 * (c_prob.argmax(1) == py_prob.argmax(1)).mean(),
                 'same label after live HMM, %': 100 * (c_label == py_label).mean(),
                 'self-test error': float(run.stdout.split(':')[1])}
    return rel, agreement

rel, agreement = check_c(model, hmm['start'], hmm['transition'])
self_test_error = agreement['self-test error']
print(pd.DataFrame({'feature': FEATURES, 'largest relative difference': rel.max(axis=0),
                    'windows differing by over 0.1%': (rel > 1e-3).mean(axis=0) * 100}).round(5).to_string(index=False))
print(agreement)

compiled with no warnings | self-test worst probability error: 2.98023e-08 | 8,558 windows in 3.5 s on the laptop
    feature  largest relative difference  windows differing by over 0.1%
   mag_mean                      0.00000                             0.0
    mag_std                      0.00000                             0.0
    mag_min                      0.00000                             0.0
    mag_max                      0.00000                             0.0
  mag_range                      0.00002                             0.0
     x_mean                      0.00000                             0.0
     y_mean                      0.00000                             0.0
     z_mean                      0.00000                             0.0
      x_std                      0.00000                             0.0
      y_std                      0.00000                             0.0
      z_std                      0.00000                             0.0
      pitc

## 5. Adding it to the firmware

Copy `export/sg_xgb.h`, `export/sg_xgb_model.h` and (optionally) `export/sg_xgb_test_vectors.h` next to the `.ino`. Then, in outline:

```c
#include "sg_xgb_test_vectors.h"           // includes sg_xgb.h and sg_xgb_model.h

static float win_x[SG_WIN], win_y[SG_WIN], win_z[SG_WIN];
static int   win_n = 0;
static sg_hmm_state hmm_state;

void setup() {
  // ... existing setup ...
  sg_hmm_reset(&hmm_state);
  float err = sg_self_test();               // about 0.5 s; should print well under 0.001
  Serial.printf("# activity model self-test: worst error %g %s\n", err, err < 1e-3f ? "OK" : "FAILED");
}

// call for every 50 Hz sample, AFTER calibration and to_model_frame() (Stage 7)
void add_sample(const float acc[3]) {
  win_x[win_n] = acc[0]; win_y[win_n] = acc[1]; win_z[win_n] = acc[2];
  if (++win_n == SG_WIN) {                  // every 30 s
    float f[SG_N_FEATURES], p[SG_N_CLASSES];
    sg_features(win_x, win_y, win_z, f);
    sg_predict_proba(f, p);
    int activity = sg_hmm_step(&hmm_state, p);   // 0 sleep, 1 sedentary, 2 light, 3 moderate-vigorous
    int moving = activity >= 2;
    // ... send / show activity ...
    win_n = 0;
  }
}
```

**Notes**
- **A missing sample** (a gap in `sample_index`): start the window again (`win_n = 0`), as Stage 0 did with gaps in Capture-24
- **The work every 30 s** is mostly the rhythm part of the features (226 frequency steps x 1500 samples). On a chip without floating-point hardware this is a few tenths of a second, run once per 30 s. Measure it on the watch with `micros()` and keep it away from the 50 Hz sampling (for example, do it in the main loop, not the sensor interrupt)
- **RAM:** 3 x 1500 floats for the window + 2 x 1500 for the cos/sin table + 1500 for the magnitude = about 42 kB of the C3's 400 kB
- **This replaces nothing yet:** the current 2.56 s still/moving forest (`har_model.h`) and the fall model keep running. Decide with the team whether this model's 30 s answers replace the old still/moving output

## 6. What this shows

**1. Size.** No model within 0.005 of the Stage 3 model's validation score fit under 200 kB (the closest, depth 4 with 200 rounds, scored 0.733 at 214 kB). So the rule's fallback picked the best model under 200 kB: **depth 3, 300 rounds: 17,658 nodes, 161 kB** in flash, against about 2 MB for the Stage 3 model.

**2. Shrinking it costs little.** On the 30 test people with the whole-day HMM: 4-class kappa **0.810**, against 0.818 for the big model. Person by person that's a tie (−0.007, 95% range −0.022 to +0.007). Still vs moving: 0.767 against 0.790.

**3. The live HMM costs a bit more.** Using only the past (as the watch must): 4-class kappa **0.782** (−0.029 against the whole-day HMM, a real difference), still vs moving 0.739. That is still far better than no HMM at all (0.686).

**4. The C code gives the same answers as Python.** Over 8,558 windows from 4 test people: every feature within 0.1% (in fact far closer), probabilities within 0.0000002, and the same final label after the live HMM for 100% of windows. The start-up self-test passes.

**Not done yet**
- **Speed on the watch itself** (no floating-point hardware) is unmeasured: time `sg_features` with `micros()` on the ESP32-C3
- **The watch's axes and calibration** (Stage 7) must be applied first, or the model reads the wrist wrongly
- **The model learned from 121 people** (so it could be tested on 30). For the final watch it could be retrained on all 151 with the same settings

In [8]:
# every claim above, checked against the numbers
g = grid.set_index(['max_depth', 'rounds'])
assert not ((grid['mean macro-F1'] >= stage3_f1 - 0.005) & (grid.kB < 200)).any()
assert round(g.loc[(4, 200), 'mean macro-F1'], 3) == 0.733 and round(g.loc[(4, 200), 'kB']) == 214
assert (DEPTH, ROUNDS) == (3, 300) and len(value) == 17658 and round(model_kb) == 161
r = results.set_index('model')
assert round(r.loc['watch XGB (small) + HMM (whole day)', '4-class kappa'], 3) == 0.810
assert round(r.loc['Stage 3 XGB (big) + HMM (whole day)', '4-class kappa'], 3) == 0.818
assert round(r.loc['watch XGB (small) + HMM (whole day)', 'binary kappa'], 3) == 0.767
assert round(r.loc['watch XGB (small) + live HMM', '4-class kappa'], 3) == 0.782
assert round(r.loc['watch XGB (small) + live HMM', 'binary kappa'], 3) == 0.739
assert round(r.loc['watch XGB (small)', '4-class kappa'], 3) == 0.686
m1, lo1, hi1 = paired_bootstrap(per['watch XGB (small) + HMM (whole day)'], per['Stage 3 XGB (big) + HMM (whole day)'])
m2, lo2, hi2 = paired_bootstrap(per['watch XGB (small) + live HMM'], per['watch XGB (small) + HMM (whole day)'])
assert lo1 < 0 < hi1 and round(m1, 3) == -0.007 and hi2 < 0 and round(m2, 3) == -0.029
assert len(Xs) == 8558 and (rel <= 1e-3).all() and agreement['largest probability difference'] < 2e-7
assert agreement['same label after live HMM, %'] == 100 and self_test_error < 1e-6
print('every claim above matches the numbers')

every claim above matches the numbers


## 7. The final watch model: trained on all 151 people

Everything above used a model trained on 121 people, so it could be **tested** on the other 30. For the watch itself, it's better to learn from **everyone**: same settings (depth 3, 300 rounds), same class weights, on the GPU. The HMM is learned again from all 151 people's labels too.

**The catch:** nobody in Capture-24 is left out to test this exact model. Its expected accuracy is the tested twin's (sections 2 and 6). Two checks instead:

- **GOTOV**, which neither model has seen (35 older people, with the axes lined up as in Stage 6): the final model should score about the same as its tested twin, or a little better
- **The C check** from section 4, run again on the final files

After this section, `export/` holds the **final** model. The tested twin stays in `results/xgb_watch.json`.

In [9]:
from common import learn_hmm

t0 = time.time()
final = fit(make_xgb(DEPTH, ROUNDS), F, y)                     # all 151 people, on the GPU
final.save_model(RESULTS_DIR / 'xgb_watch_all151.json')
hmm_start_all, hmm_trans_all = learn_hmm(y, pid)               # rows are in time order within each person
np.savez(RESULTS_DIR / 'hmm_all151.npz', start=hmm_start_all, transition=hmm_trans_all)
print(f'trained on {len(y):,} windows from {len(np.unique(pid))} people in {time.time() - t0:.0f} s')
print('HMM change from the 121-person one: start chances by up to', round(float(np.abs(hmm_start_all - hmm['start']).max()), 4),
      '| switching table by up to', round(float(np.abs(hmm_trans_all - hmm['transition']).max()), 4))

# independent check on GOTOV (neither model has seen it), axes lined up as in Stage 6
s5 = np.load(RESULTS_DIR / 'stage5_gotov_predictions.npz')
Fg = np.load('processed/gotov/features_rotated.npz')['F']
to3 = lambda v: np.maximum(np.asarray(v), 1) - 1
rows = []
for name, m_, st, tr_ in [('tested twin (121 people)', model, hmm['start'], hmm['transition']),
                          ('final (151 people)', final, hmm_start_all, hmm_trans_all)]:
    pr = m_.predict_proba(Fg)
    for hmm_name, pred in [('whole-day HMM', smooth(pr, s5['pid'], st, tr_)), ('live HMM', live_hmm(pr, s5['pid'], st, tr_))]:
        rows.append({'model': name, 'HMM': hmm_name, 'binary kappa': scores(to_binary(s5['y']), to_binary(pred), 2)['kappa'],
                     '3-class kappa': scores(to3(s5['y']), to3(pred), 3)['kappa']})
gotov_check = pd.DataFrame(rows)
gotov_check.to_csv(RESULTS_DIR / 'stage8_final_gotov_check.csv', index=False)
print('\nGOTOV (35 older people, never seen):')
print(gotov_check.round(3).to_string(index=False))

trained on 301,164 windows from 151 people in 3 s
HMM change from the 121-person one: start chances by up to 0.0083 | switching table by up to 0.0014

GOTOV (35 older people, never seen):
                   model           HMM  binary kappa  3-class kappa
tested twin (121 people) whole-day HMM         0.972          0.799
tested twin (121 people)      live HMM         0.932          0.805
      final (151 people) whole-day HMM         0.968          0.798
      final (151 people)      live HMM         0.929          0.808


In [10]:
tab_final = tables_from(final, ROUNDS)
final_kb = write_header(tab_final, hmm_start_all, hmm_trans_all,
                        'FINAL: trained on all 151 Capture-24 people (no one held out). Tested twin (121 people): results/xgb_watch.json')
write_test_vectors(final)
rel_final, agreement_final = check_c(final, hmm_start_all, hmm_trans_all)
print(f"final model: {len(tab_final['value']):,} nodes, {final_kb:.0f} kB in flash (tested twin: {len(value):,} nodes, {model_kb:.0f} kB)")
print(agreement_final)
assert final_kb < 200 and agreement_final['same label after live HMM, %'] == 100 and (rel_final <= 1e-3).all()
# WHY not exactly equal: C works out the features in float (about 0.00002 off from Python's double); now and then a
# feature sits right on a tree's threshold and goes the other way. A small, rare difference is expected.
assert agreement_final['largest probability difference'] < 0.01 and agreement_final['self-test error'] < 1e-6
first_lines = (EXPORT / 'sg_xgb_model.h').read_text().splitlines()[:3]
assert 'FINAL' in first_lines[2]
print('\nexport/sg_xgb_model.h now says:', *first_lines, sep='\n  ')

compiled with no warnings | self-test worst probability error: 5.96046e-08 | 8,558 windows in 3.4 s on the laptop
final model: 17,896 nodes, 163 kB in flash (tested twin: 17,658 nodes, 161 kB)
{'largest feature difference (relative)': 1.635441905384709e-05, 'largest probability difference': 0.003990098834037781, 'windows differing by over 0.0001': 3, 'same top class (no HMM), %': np.float64(100.0), 'same label after live HMM, %': np.float64(100.0), 'self-test error': 5.96046e-08}

export/sg_xgb_model.h now says:
  // GENERATED by stage8_export_xgboost.ipynb. Do not edit: re-run the notebook.
  // XGBoost depth 3, 300 rounds x 4 classes = 1200 trees, 17896 nodes.
  // FINAL: trained on all 151 Capture-24 people (no one held out). Tested twin (121 people): results/xgb_watch.json


## 8. The final model, in short

- **Trained on all 151 people**, same settings: 17,896 nodes, **163 kB** in flash (the tested twin: 17,658 nodes, 161 kB). The HMM learned from all 151 differs from the 121-person one by under 0.01
- **On GOTOV (never seen by either)** the final model scores the same as its tested twin: still vs moving 0.968 vs 0.972, 3-class 0.798 vs 0.799 (whole-day HMM); 0.929 vs 0.932 and 0.808 vs 0.805 with the live HMM. So learning from 30 more people didn't help or hurt here, and the tested twin's scores (section 2) are the best guide to this model's accuracy
- **The C files in `export/` are now the final model**, and the C check passes: same label after the live HMM for 100% of 8,558 windows. A handful of windows differ slightly in probability (largest 0.004), because C computes features in float: a feature that sits right on a tree's threshold can go the other way

In [11]:
ga = gotov_check.set_index(['model', 'HMM'])
tw, fi = 'tested twin (121 people)', 'final (151 people)'
assert len(tab_final['value']) == 17896 and round(final_kb) == 163
assert max(np.abs(hmm_start_all - hmm['start']).max(), np.abs(hmm_trans_all - hmm['transition']).max()) < 0.01
assert [round(ga.loc[(fi, 'whole-day HMM'), k], 3) for k in ['binary kappa', '3-class kappa']] == [0.968, 0.798]
assert [round(ga.loc[(tw, 'whole-day HMM'), k], 3) for k in ['binary kappa', '3-class kappa']] == [0.972, 0.799]
assert [round(ga.loc[(fi, 'live HMM'), k], 3) for k in ['binary kappa', '3-class kappa']] == [0.929, 0.808]
assert [round(ga.loc[(tw, 'live HMM'), k], 3) for k in ['binary kappa', '3-class kappa']] == [0.932, 0.805]
assert agreement_final['same label after live HMM, %'] == 100 and round(agreement_final['largest probability difference'], 3) == 0.004
print('every claim above matches the numbers; windows differing by over 0.0001:', agreement_final['windows differing by over 0.0001'], 'of', len(Xs))

every claim above matches the numbers; windows differing by over 0.0001: 3 of 8558
